In [13]:
import os
import sys
sys.path.append("..")  # add project root
from prog import mlps, featlib, trainer, hlprs
import Datasets.matconv as mc
import torch
import matplotlib.pyplot as plt
import numpy as np

### Testing Symnet for Hardcoded Accuracy vs autodiff wrt $t$

    - Use MLP with activation to create regularized fit of data. 
    - Create Symnet object with hardcoded entries 
    - Use generated input tensor F to compare ut over same batch. 

In [14]:
device='cpu'

### Data Settings

In [15]:
# Data generation settings
noise=0.0
nu=0.02
stride_t=1
stride_x=1
part_num=1
which_part=1
seed=1432

In [16]:
# Dataset building

partitions = mc.build_dataset_from_burgers(noise_level=noise, nu=nu, stride_t=stride_t, stride_x=stride_x, seed=seed, quantile_splits=part_num, return_partitions=True)
t_np, x_np, y_np, y_noisy_np, N = partitions[[k for k in partitions if k.startswith(f'Q{which_part}:')][0]]
# batches are generated over t_np, x_np, y_np, y_noisy_np

t_torch       = torch.from_numpy(t_np).to(device)
x_torch       = torch.from_numpy(x_np).to(device)
y_torch       = torch.from_numpy(y_np).to(device)
y_noisy_torch = torch.from_numpy(y_noisy_np).to(device)

In [31]:
for i in range(len(x_torch)):
    if x_torch[i] == 0.0:
        print("t[{}] = 0.0".format(i))

t[0] = 0.0
t[256] = 0.0
t[512] = 0.0
t[768] = 0.0
t[1024] = 0.0
t[1280] = 0.0
t[1536] = 0.0
t[1792] = 0.0
t[2048] = 0.0
t[2304] = 0.0
t[2560] = 0.0
t[2816] = 0.0
t[3072] = 0.0
t[3328] = 0.0
t[3584] = 0.0
t[3840] = 0.0
t[4096] = 0.0
t[4352] = 0.0
t[4608] = 0.0
t[4864] = 0.0
t[5120] = 0.0
t[5376] = 0.0
t[5632] = 0.0
t[5888] = 0.0
t[6144] = 0.0
t[6400] = 0.0
t[6656] = 0.0
t[6912] = 0.0
t[7168] = 0.0
t[7424] = 0.0
t[7680] = 0.0
t[7936] = 0.0
t[8192] = 0.0
t[8448] = 0.0
t[8704] = 0.0
t[8960] = 0.0
t[9216] = 0.0
t[9472] = 0.0
t[9728] = 0.0
t[9984] = 0.0
t[10240] = 0.0
t[10496] = 0.0
t[10752] = 0.0
t[11008] = 0.0
t[11264] = 0.0
t[11520] = 0.0
t[11776] = 0.0
t[12032] = 0.0
t[12288] = 0.0
t[12544] = 0.0
t[12800] = 0.0
t[13056] = 0.0
t[13312] = 0.0
t[13568] = 0.0
t[13824] = 0.0
t[14080] = 0.0
t[14336] = 0.0
t[14592] = 0.0
t[14848] = 0.0
t[15104] = 0.0
t[15360] = 0.0
t[15616] = 0.0
t[15872] = 0.0
t[16128] = 0.0
t[16384] = 0.0
t[16640] = 0.0
t[16896] = 0.0
t[17152] = 0.0
t[17408] = 0.0
t[17664] = 

#### Training Settings

Loss is calculated like
$$ L[\tilde{u}] = \lambda_{data}||\tilde{u}-u_{raw}||_{L2}+\lambda_{pde}||\tilde{u}_t - Sym(\tilde{u},\tilde{u_x},\tilde{u_{xx}})||_{mse}+\lambda_{reg}||w_{symnet}||+\lambda_{tv}||\tilde{u}||_{tv}$$

In [75]:
selected_derivs=('u','u_x','u_xx') # up to first order derivative inputs
lr=0.001
batch_size=1000  # number of data points used on optimization step 
                 # data points are generated according to tensor.randint over spacetime mesh
lam_pde=0.0
lam_reg=0.0
lam_tv=0.0
lam_data=10.0

#output settings
log_every=1000

In [ ]:
# Initializing u_model and a dummy v_model so that trainer works
u_model=mlps.SimpleMLP(n_layers=4,hidden_size=64)
v_model=mlps.EQL(in_dim=len(selected_derivs), prod_dim=0, bias=False) # dummy v_model for working code
for param in v_model.parameters(): #freeze v_model params
    param.requires_grad = False
print('v model parameters (weights) are not receiving gradients (improve data fit accuracy)')

train_config=trainer.TrainerConfig(
    lr=lr,
    lambda_pde=lam_pde, 
    lambda_reg=lam_reg,
    lambda_tv=lam_tv, 
    lambda_data=lam_data, 
    selected_derivs=selected_derivs, 
)
ft = featlib.FeatureTensor(selected_derivs, normalize=False)
feature_builder = ft.build
train=trainer.PDETrainer(u_model=u_model,v_model=v_model,cfg=train_config, feature_builder=feature_builder)

v model parameters (weights) are not receiving gradients (improve data fit accuracy)


C:\Users\sami\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\nn\init.py:582: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


'rmrfjk'

### Training Loop

#### MLP gets trained over raw data

In [77]:
for i in range(4000): 
    t,x,u_noisy,u_clean = hlprs.make_nonlinear_batch(batch_size=batch_size,t_torch=t_torch,x_torch=x_torch,y_torch=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    ld = out['loss_data']
    lp = out['loss_pde']
    if i%log_every==0:
        print(f'step {i} data_loss={ld} pde_loss={lp}')

step 0 data_loss=0.46915295720100403 pde_loss=0.0905875414609909


KeyboardInterrupt: 

### Test 2: Hardcoded Symnet weights and u_model pretrained on data

In [ ]:
symnet = mlps.EQL(in_dim=3, prod_dim=2, bias=False)
symnet.linear.weight
with torch.no_grad():
    symnet.linear.weight.copy_(torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=symnet.linear.weight.dtype, device=symnet.linear.weight.device))
    symnet.readout.weight.copy_(torch.tensor([[0.0, 0, 0.02, -1.00]], dtype=symnet.readout.weight.dtype, device=symnet.readout.weight.device))
print("symnet hardcoded weights:")
print(symnet.readout.weight, '\n')
print(symnet.linear.weight)

symnet hardcoded weights:
Parameter containing:
tensor([[ 0.0000,  0.0000,  0.0200, -1.0000]], requires_grad=True) 

Parameter containing:
tensor([[1., 0., 0.],
        [0., 1., 0.]], requires_grad=True)


We compare u_t over current batch with Symnet. This demonstrates the result of $$L_{pde} = \lambda_{pde}||u_t - Symnet||$$

In [ ]:
# Check accuracy of hardcoded Symnet
symnet_error = train.mse(train.u_t,symnet(train.F))
print(f'SYMNET error vs u_t (autograd) with hardcoded parameters (forced selected pde): \n{symnet_error}')

SYMNET error vs u_t (autograd) with hardcoded parameters (forced selected pde): 
0.0084390165284276


In [ ]:
train.v = symnet # swap out v_model for symnet
print(train.v.linear.weight) # verify weights are hardcoded
params = list(train.u.parameters()) + list(train.v.parameters())
train.optimizer = torch.optim.Adam(params, lr=lr) # update optimizer to replace dummy v model with symnet weights


Parameter containing:
tensor([[1., 0., 0.],
        [0., 1., 0.]], requires_grad=True)


In [ ]:
# Freeze mlp for data fit, regularizer on data term to zero, loss only depends on pde loss 
# Compare loss values above 

train_config.lambda_data = 0.0
train_config.lambda_pde = 0.5
for params in u_model.parameters(): 
    params.requires_grad = False

for params in symnet.parameters(): 
    params.requires_grad = True

In [ ]:
plot_pde_ls = []

for i in range(10000): 
    t,x,u_noisy,u_clean = hlprs.make_nonlinear_batch(batch_size=batch_size,t_torch=t_torch,x_torch=x_torch,y_torch=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    ld = out['loss_data']
    lp = out['loss_pde']
    plot_pde_ls.append(lp)
    if i%log_every==0:
        print(f'step_{i}, data_loss={ld}, pde_loss={lp}')

step_0, data_loss=9.997782035497949e-05, pde_loss=0.007186426315456629
step_1000, data_loss=0.0001096274791052565, pde_loss=0.003615459892898798
step_2000, data_loss=9.525108180241659e-05, pde_loss=0.0035742269828915596
step_3000, data_loss=0.00011325559898978099, pde_loss=0.003739277832210064
step_4000, data_loss=0.00012826680904254317, pde_loss=0.00331898988224566
step_5000, data_loss=0.00011258759332122281, pde_loss=0.0037998121697455645
step_6000, data_loss=8.918944513425231e-05, pde_loss=0.0030206269584596157
step_7000, data_loss=8.906295988708735e-05, pde_loss=0.003272505011409521
step_8000, data_loss=0.00012443686136975884, pde_loss=0.0036954418756067753
step_9000, data_loss=0.00012023788440274075, pde_loss=0.0035609391052275896


In [ ]:
symnet.linear.weight, symnet.readout.weight

(Parameter containing:
 tensor([[ 1.0310,  0.0134,  0.0160],
         [-0.0162,  0.9948,  0.0029]], requires_grad=True),
 Parameter containing:
 tensor([[-0.0220, -0.0149,  0.0021, -0.9742]], requires_grad=True))

Q: How do we know symnet is receiving gradients? 

### Test 3: random initialization of symnet with u_model pretrained on data

In [ ]:
symnet = mlps.EQL(in_dim=3, prod_dim=2, bias=False)
symnet.linear.weight

Parameter containing:
tensor([[-0.0098, -0.5588, -0.1601],
        [ 0.3478,  0.3774, -0.5215]], requires_grad=True)

In [ ]:
train.v = symnet # swap out v_model for symnet
print(train.v.linear.weight) # verify weights are hardcoded
params = list(train.u.parameters()) + list(train.v.parameters())
train.optimizer = torch.optim.Adam(params, lr=lr) # update optimizer to replace dummy v model with symnet weights

Parameter containing:
tensor([[-0.0098, -0.5588, -0.1601],
        [ 0.3478,  0.3774, -0.5215]], requires_grad=True)


In [ ]:
#Sanity check: set loss to L_p, and freeze the data fit while training only symnet

train_config.lambda_data = 0.0
train_config.lambda_pde = 100.0
for params in u_model.parameters(): 
    params.requires_grad = False

for params in symnet.parameters(): 
    params.requires_grad = True

In [ ]:
plot_pde_ls = []

for i in range(4000): 
    t,x,u_noisy,u_clean = hlprs.make_nonlinear_batch(batch_size=batch_size,t_torch=t_torch,x_torch=x_torch,y_torch=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    ld = out['loss_data']
    lp = out['loss_pde']
    plot_pde_ls.append(lp)
    if i%log_every==0:
        print(f'step_{i}, data_loss={ld}, pde_loss={lp}')

step_0, data_loss=0.00012520495511125773, pde_loss=2.7801637649536133
step_1000, data_loss=0.00010012510028900579, pde_loss=0.19140776991844177
step_2000, data_loss=0.00010357212886447087, pde_loss=0.12209688127040863
step_3000, data_loss=0.00012348525342531502, pde_loss=0.003928862512111664


In [ ]:
print(list(train.v.parameters()))

[Parameter containing:
tensor([[ 0.0177, -1.0617, -0.0031],
        [ 2.8062,  0.0355,  0.0436]], requires_grad=True), Parameter containing:
tensor([[-0.0227, -0.0151,  0.0015,  0.3348]], requires_grad=True)]


In [ ]:
2.81*1.06*0.335


1.000428

### Test 4: Random init of Symnet and Data fitting MLP

In [ ]:
from torch import nn

In [ ]:
mat  = nn.Linear(3,2)
matr = nn.Linear(3,1)
maT  = mat.weight.T
col1 = maT[:,0]
col2 = maT[:,1]
torch.outer(col1,col2)
matr.weight

tensor(-0.2287, grad_fn=<SelectBackward0>)

In [ ]:
u_model = mlps.SimpleMLP(n_layers=4,hidden_size=64)
symnet = mlps.EQL(in_dim=3, prod_dim=2, bias=False)
symnet.linear.weight

Parameter containing:
tensor([[-0.3968, -0.4257,  0.4625],
        [ 0.4068,  0.0931,  0.3187]], requires_grad=True)

In [ ]:
train.u = u_model # swap out u_model (data fitting)
train.v = symnet # swap out v_model for symnet
print(train.v.linear.weight) 
params = list(train.u.parameters()) + list(train.v.parameters())
train.optimizer = torch.optim.Adam(params, lr=lr) # update optimizer to replace dummy v model with symnet weights


Parameter containing:
tensor([[ 0.5215,  0.3349, -0.4994],
        [ 0.1136, -0.3286,  0.4763]], requires_grad=True)


In [ ]:
train_config.lambda_data = 10.0
train_config.lambda_pde = 0.5
for params in u_model.parameters(): 
    params.requires_grad = True

for params in symnet.parameters(): 
    params.requires_grad = True

In [ ]:
plot_pde_ls = []

for i in range(4000): 
    t,x,u_noisy,u_clean = hlprs.make_nonlinear_batch(batch_size=batch_size,t_torch=t_torch,x_torch=x_torch,y_torch=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    ld = out['loss_data']
    lp = out['loss_pde']
    plot_pde_ls.append(lp)
    if i%log_every==0:
        print(f'step_{i}, data_loss={ld}, pde_loss={lp}')

# Note: 8k epochs

step_0, data_loss=0.44073137640953064, pde_loss=0.0031936822924762964
step_1000, data_loss=0.002242055255919695, pde_loss=0.007500870618969202
step_2000, data_loss=0.0004450723936315626, pde_loss=0.0032792368438094854
step_3000, data_loss=0.00015983180492185056, pde_loss=0.0013032418210059404


In [ ]:
print(list(train.v.parameters()))

[Parameter containing:
tensor([[ 1.5507,  0.0055,  0.0214],
        [ 0.0047, -0.8972, -0.0049]], requires_grad=True), Parameter containing:
tensor([[-0.0074, -0.0043,  0.0061,  0.7213]], requires_grad=True)]


In [ ]:
train.v.readout.weight[0,-1].item()

0.7212913632392883

In [ ]:
ww1 = train.v.linear.weight[0]
ww2 = train.v.linear.weight[1]
print(ww1,ww2)
out = torch.outer(ww1,ww2)   # (n, n)
print(out)
out*train.v.readout.weight[0,-1].item()

tensor([1.5507, 0.0055, 0.0214], grad_fn=<SelectBackward0>) tensor([ 0.0047, -0.8972, -0.0049], grad_fn=<SelectBackward0>)
tensor([[ 7.3587e-03, -1.3913e+00, -7.5403e-03],
        [ 2.6156e-05, -4.9452e-03, -2.6802e-05],
        [ 1.0136e-04, -1.9165e-02, -1.0387e-04]], grad_fn=<MulBackward0>)


tensor([[ 5.3078e-03, -1.0035e+00, -5.4388e-03],
        [ 1.8866e-05, -3.5670e-03, -1.9332e-05],
        [ 7.3114e-05, -1.3823e-02, -7.4919e-05]], grad_fn=<MulBackward0>)